In [1]:
import pandas as pd
import yfinance as yf
from bcb import sgs
import datetime
import ipeadatapy as ipea
import requests
import duckdb
import os
import glob
import json

# Ingestão

## CONAB

In [43]:
df_custo_producao= pd.read_csv('d:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\custo_produção.txt', sep=';')
df_estimativa_graos = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\estimativa_graos.txt', sep=';')
df_estoque_publico = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\estoques_publicos.txt', sep=';')
df_frete = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab/frete.txt', sep=';')
df_oferta_demanda = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\oferta_e_demanda.txt', sep=';')
df_preco_uf_mensal = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\preco_agropecuaria_mensal_uf.txt', sep=';')
df_preco_municipio_mensal = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\preco_agropecuraia_mensal_municipio.txt', sep=';')
df_serie_hitorica_graos = pd.read_csv('D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Conab\serie_historica_graos.txt', sep=';')

## Yahoo Finance (CBOT Arroz)

In [44]:
# 1. Ingestão da API
ticker_arroz = "ZR=F"
arroz = yf.Ticker(ticker_arroz)
df_CBOT = arroz.history(period="max")

# 2. Transforma o índice (Date) em uma coluna comum do DataFrame
df_CBOT = df_CBOT.reset_index()

# 3. Tratamentos para a camada Staging:
# - Renomeia a coluna 'Date' para 'data'
# - Converte para o tipo DATE nativo (removendo fuso horário/horário)
df_CBOT["Date"] = pd.to_datetime(df_CBOT["Date"]).dt.date
df_CBOT = df_CBOT.rename(columns={"Date": "data"})

# Exibe as primeiras linhas
df_CBOT.head()

,data,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1999-09-14,6.4095,6.4095,6.4095,6.4095,0,0.0,0.0
1,1999-09-16,6.4095,6.4095,6.4095,6.4650,0,0.0,0.0
2,1999-09-17,6.4650,6.4095,6.4095,6.4650,0,0.0,0.0
3,1999-09-20,6.4650,6.4095,6.4095,6.4115,0,0.0,0.0
4,1999-09-21,6.4115,6.4095,6.4095,6.3765,0,0.0,0.0


## CEPEA (Valor Histórico Arroz Casca)

In [45]:
caminho_arquivo_CEPEA = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\CEPEA_20260628112648.xlsx"

In [46]:
df_cepea = pd.read_excel(caminho_arquivo_CEPEA)

## Cooperativa (Valor Arroz)

In [47]:
caminho_arquivo_cooperativa = r"D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Coperativa\coperativa.xlsx"

In [48]:
df_cooperativa = pd.read_excel(caminho_arquivo_cooperativa)

## BCB (Valor Diesel)

In [49]:
# 1. Ingestão da API do BCB
df_diesel = sgs.get({'diesel': 1393}, start='2005-01-01')

# 2. Transforma o índice (Date) em uma coluna comum do DataFrame
df_diesel = df_diesel.reset_index()

# 3. Tratamentos e padronização para a Staging:
# - Renomeia 'Date' (ou 'data') para padronizar
df_diesel = df_diesel.rename(columns={
    "Date": "data",
    "diesel": "vlr_diesel"
})

# - Garante a conversão para o tipo DATE nativo (YYYY-MM-DD)
df_diesel["data"] = pd.to_datetime(df_diesel["data"]).dt.date

# Visualizar o resultado
print(df_diesel.head())

         data  vlr_diesel
0  2005-01-01         281
1  2005-02-01         304
2  2005-03-01         314
3  2005-04-01         302
4  2005-05-01         299


## BCB (Valor Dolar)

In [50]:
data_inicio = "01-01-2005"
data_fim = datetime.datetime.now().strftime("%m-%d-%Y")

In [51]:
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$top=100000&$format=json"
)

print("⏳ Buscando histórico do dólar desde 2005 via Banco Central...")

⏳ Buscando histórico do dólar desde 2005 via Banco Central...


In [52]:
print("⏳ Buscando histórico do dólar desde 2005 via Banco Central...")

# 2. Leitura do JSON retornado pela API
df_raw_dolar = pd.read_json(url)
df_dolar = pd.DataFrame(df_raw_dolar["value"].tolist())

# 3. Tratar e renomear colunas
df_dolar = df_dolar.rename(
    columns={
        "cotacaoCompra": "valor_baixo",
        "cotacaoVenda": "valor_atual",
        "dataHoraCotacao": "data_cotacao",
    }
)

# Criando colunas de metadados
df_dolar["moeda"] = "USD"
df_dolar["nome"] = "Dólar Americano/Real Brasileiro"
df_dolar["valor_alto"] = df_dolar["valor_atual"]  # Usando a venda como referência
df_dolar["data_registro"] = datetime.datetime.now()

# 4. Ajuste de tipos e reordenação das colunas
df_dolar["data_cotacao"] = pd.to_datetime(df_dolar["data_cotacao"])

colunas_ordenadas = [
    "moeda",
    "nome",
    "valor_alto",
    "valor_baixo",
    "valor_atual",
    "data_cotacao",
    "data_registro",
]
df_dolar = df_dolar[colunas_ordenadas]

# 5. Ordenar por data (da mais antiga para a mais recente) - Corrigido df para df_dolar
df_dolar = df_dolar.sort_values(by="data_cotacao").reset_index(drop=True)

print(f"✅ Sucesso! DataFrame criado com {len(df_dolar)} linhas.")

⏳ Buscando histórico do dólar desde 2005 via Banco Central...
✅ Sucesso! DataFrame criado com 5453 linhas.


## IPEA (Inflação)

In [53]:
#print("\n⏳ Buscando índices de inflação via API do IPEA...")

# Códigos oficiais das séries de variação mensal (%) no IPEA:
# IPCA: "PRECOS12_IPCAM12" ou número de índice. Vamos buscar as variações mensais.
# Para garantir compatibilidade total sem depender do pacote pesado, usamos a API deles direto via Pandas:

#url_ipca = "https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='PRECOS12_IPCA12')"
#url_igpm = "https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='IGP12_IGPM12')"

# Capturando IPCA (Número índice ou taxa)
#df_ipca_raw = pd.read_json(url_ipca)
#df_ipca = pd.DataFrame(df_ipca_raw["value"].tolist())

# Capturando IGP-M
#df_igpm_raw = pd.read_json(url_igpm)
#df_igpm = pd.DataFrame(df_igpm_raw["value"].tolist())

# Tratando o DataFrame do IPCA
#df_ipca = df_ipca[["VALDATA", "VALVALOR"]].rename(columns={"VALDATA": "data", "VALVALOR": "ipca"})
# Forçamos utc=True para o Pandas reconhecer o fuso da API, e depois limpamos com .tz_localize(None)
#df_ipca["data"] = pd.to_datetime(df_ipca["data"], utc=True).dt.tz_localize(None)

# Tratando o DataFrame do IGP-M
#df_igpm = df_igpm[["VALDATA", "VALVALOR"]].rename(columns={"VALDATA": "data", "VALVALOR": "igpm"})
#df_igpm["data"] = pd.to_datetime(df_igpm["data"], utc=True).dt.tz_localize(None)

# 3. Cruzar os dados de inflação em um único DataFrame e filtrar a partir de 2005
#df_inflacao = pd.merge(df_ipca, df_igpm, on="data", how="outer")
#df_inflacao = df_inflacao[df_inflacao["data"] >= "2005-01-01"].sort_values("data").reset_index(drop=True)

#print("✅ Dados de Inflação carregados com sucesso!")

In [54]:
print("\n⏳ Buscando índices de inflação via API do IPEA...")

# 1. Ajuste das URLs para HTTPS
url_ipca = (
    "https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='PRECOS12_IPCA12')"
)
url_igpm = (
    "https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='IGP12_IGPM12')"
)

# 2. Definição do User-Agent para evitar bloqueios
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


def buscar_dados_ipea(url):
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()  # Lança exceção em caso de erro HTTP
    dados = response.json()
    return pd.DataFrame(dados["value"])


try:
    # Capturando IPCA
    df_ipca = buscar_dados_ipea(url_ipca)
    df_ipca = df_ipca[["VALDATA", "VALVALOR"]].rename(
        columns={"VALDATA": "data", "VALVALOR": "ipca"}
    )
    df_ipca["data"] = pd.to_datetime(df_ipca["data"], utc=True).dt.tz_localize(
        None
    )

    # Capturando IGP-M
    df_igpm = buscar_dados_ipea(url_igpm)
    df_igpm = df_igpm[["VALDATA", "VALVALOR"]].rename(
        columns={"VALDATA": "data", "VALVALOR": "igpm"}
    )
    df_igpm["data"] = pd.to_datetime(df_igpm["data"], utc=True).dt.tz_localize(
        None
    )

    # 3. Cruzar os dados e filtrar
    df_inflacao = pd.merge(df_ipca, df_igpm, on="data", how="outer")
    df_inflacao = (
        df_inflacao[df_inflacao["data"] >= "2005-01-01"]
        .sort_values("data")
        .reset_index(drop=True)
    )

    print("✅ Dados de Inflação carregados com sucesso!")

except requests.exceptions.RequestException as e:
    print(f"❌ Erro ao conectar com a API do IPEA: {e}")


⏳ Buscando índices de inflação via API do IPEA...
✅ Dados de Inflação carregados com sucesso!


# Camada Raw

In [55]:
con = duckdb.connect()

In [56]:
con.register("raw_cepea", df_cepea)
con.register("raw_cbot", df_CBOT)
con.register("raw_cooperativa", df_cooperativa)
con.register("raw_diesel", df_diesel)
con.register("raw_dolar", df_dolar)
con.register("raw_inflacao", df_inflacao)

con.register("raw_conab_custo_producao", df_custo_producao)
con.register("raw_conab_estimativa_graos", df_estimativa_graos)
con.register("raw_conab_estoque_publico", df_estoque_publico)
con.register("raw_conab_frete", df_frete)
con.register("raw_conab_oferta_demanda", df_oferta_demanda)
con.register("raw_conab_preco_uf_mensal", df_preco_uf_mensal)
con.register("raw_conab_preco_municipio_mensal", df_preco_municipio_mensal)
con.register("raw_conab_serie_historica_graos", df_serie_hitorica_graos)

In [57]:
con.execute("""
    SELECT DISTINCT produto
    FROM raw_conab_custo_producao
    where produto like 'A%';
""").df()


,produto
0,ABACAXI
1,AMENDOA DE ANDIROBA
2,ARROZ
3,ALHO
4,ALGODAO EM PLUMA
5,ACAI
6,AMENDOA DE BARU


In [58]:
con.execute("""
    SELECT *
    FROM raw_cbot
""").df()

,data,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1999-09-14,6.4095,6.4095,6.4095,6.4095,0,0.0,0.0
1,1999-09-16,6.4095,6.4095,6.4095,6.4650,0,0.0,0.0
2,1999-09-17,6.4650,6.4095,6.4095,6.4650,0,0.0,0.0
3,1999-09-20,6.4650,6.4095,6.4095,6.4115,0,0.0,0.0
4,1999-09-21,6.4115,6.4095,6.4095,6.3765,0,0.0,0.0
...,...,...,...,...,...,...,...,...
6766,2026-09-14,15.6550,15.6550,15.5500,15.5650,462,0.0,0.0
6767,2026-09-15,15.9100,15.9850,15.7800,15.7850,613,0.0,0.0
6768,2026-09-16,15.7700,15.9050,15.7050,15.8400,650,0.0,0.0
6769,2026-09-17,15.8300,15.9350,15.7650,15.7900,321,0.0,0.0


In [59]:
diretorio_destino = r'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw'
views_para_exportar = {
    "raw_cepea": "cepea_raw.parquet",
    "raw_cbot": "cbot_raw.parquet",
    "raw_cooperativa": "cooperativa_raw.parquet",
    "raw_diesel": "diesel_raw.parquet",
    "raw_dolar": "dolar_raw.parquet",
    "raw_inflacao": "inflacao_raw.parquet",
    "raw_conab_custo_producao": "conab_custo_producao_raw.parquet",
    "raw_conab_estimativa_graos": "conab_estimativa_graos_raw.parquet",
    "raw_conab_estoque_publico": "conab_estoque_publico_raw.parquet",
    "raw_conab_frete": "conab_frete_raw.parquet",
    "raw_conab_oferta_demanda": "conab_oferta_demanda_raw.parquet",
    "raw_conab_preco_uf_mensal": "conab_preco_uf_mensal_raw.parquet",
    "raw_conab_preco_municipio_mensal": "conab_preco_municipio_mensal_raw.parquet",
    "raw_conab_serie_historica_graos": "conab_serie_historica_graos_raw.parquet"
}

In [60]:
for view_name, file_name in views_para_exportar.items():
    # Agora o caminho_arquivo inclui o nome do arquivo individual!
    caminho_arquivo = os.path.join(diretorio_destino, file_name).replace("\\", "/")
    
    query = f"COPY {view_name} TO '{caminho_arquivo}' (FORMAT PARQUET, COMPRESSION 'ZSTD');"
    con.execute(query)
    print(f"Exportada view '{view_name}' -> {file_name}")

print("\nProcesso concluído! Arquivos Parquet gerados com sucesso.")

Exportada view 'raw_cepea' -> cepea_raw.parquet
Exportada view 'raw_cbot' -> cbot_raw.parquet
Exportada view 'raw_cooperativa' -> cooperativa_raw.parquet
Exportada view 'raw_diesel' -> diesel_raw.parquet
Exportada view 'raw_dolar' -> dolar_raw.parquet
Exportada view 'raw_inflacao' -> inflacao_raw.parquet
Exportada view 'raw_conab_custo_producao' -> conab_custo_producao_raw.parquet
Exportada view 'raw_conab_estimativa_graos' -> conab_estimativa_graos_raw.parquet
Exportada view 'raw_conab_estoque_publico' -> conab_estoque_publico_raw.parquet
Exportada view 'raw_conab_frete' -> conab_frete_raw.parquet
Exportada view 'raw_conab_oferta_demanda' -> conab_oferta_demanda_raw.parquet
Exportada view 'raw_conab_preco_uf_mensal' -> conab_preco_uf_mensal_raw.parquet
Exportada view 'raw_conab_preco_municipio_mensal' -> conab_preco_municipio_mensal_raw.parquet
Exportada view 'raw_conab_serie_historica_graos' -> conab_serie_historica_graos_raw.parquet

Processo concluído! Arquivos Parquet gerados com 

# Camada Stagging

In [61]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\cepea_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cepea_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df["Data"] = pd.to_datetime(df["Data"], format="%d/%m/%Y").dt.date
df = df.rename(columns={
    'Data': 'date',
    'À vista R$': 'valor_BRL',
    'À vista US$': 'valor_US'
})
df.to_parquet(caminho_stagging, index=False)

##duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cepea_stagging.parquet'").df()
df.head()

,date,valor_BRL,valor_US
0,2005-06-30,20.41,8.71
1,2005-07-01,20.41,8.71
2,2005-07-03,21.18,9.04
3,2005-07-04,22.18,9.47
4,2005-07-05,20.41,8.71


In [62]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\cbot_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cbot_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df.drop(columns=["Close","Low","Dividends","Stock Splits","Open"])
df = df.rename(columns={"High": "value",
                        'data': 'date',
                         "Volume": "volume"})
df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cbot_stagging.parquet'").df()
df.head()

,date,value,volume
0,1999-09-14,6.4095,0
1,1999-09-16,6.4095,0
2,1999-09-17,6.4095,0
3,1999-09-20,6.4095,0
4,1999-09-21,6.4095,0


In [63]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\cooperativa_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cooperativa_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df["Date From"] = pd.to_datetime(df["Date From"]).dt.date
df["Date To"] = pd.to_datetime(df["Date To"]).dt.date
df = df.rename(columns={'Date From': 'date_from',
                        'Date To': 'date_to',
                        'Valor Categoria 1': 'valor_cat1',
                        'Valor Categoria 2': 'valor_cat2',
                        'Valor Categoria 3': 'calor_cat3',
                        '% no mes': 'proc_mes'})

df.to_parquet(caminho_stagging, index=False)


duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\cooperativa_stagging.parquet'").df()
#df.head()

,column_name,column_type,null,key,default,extra
0,date_from,DATE,YES,None,None,None
1,date_to,DATE,YES,None,None,None
2,valor_cat1,BIGINT,YES,None,None,None
3,valor_cat2,BIGINT,YES,None,None,None
4,calor_cat3,DOUBLE,YES,None,None,None
5,proc_mes,BIGINT,YES,None,None,None


In [64]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\diesel_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\diesel_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df.rename(columns={'data': 'date',
                        'vlr_diesel': 'value'}
)

df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\diesel_stagging.parquet'").df()
print(json.dumps(list(df.columns)))

["date", "value"]


In [65]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\dolar_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\dolar_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df.drop(columns=["moeda","nome","data_registro","valor_baixo","valor_atual"])
df["data_cotacao"] = pd.to_datetime(df["data_cotacao"]).dt.date
df = df.rename(columns={"valor_alto": "value",
                         "data_cotacao": "date"})
df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\dolar_stagging.parquet'").df()
print(json.dumps(list(df.columns)))

["value", "date"]


In [66]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\inflacao_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\inflacao_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df["data"] = pd.to_datetime(df["data"]).dt.date
df = df.rename(columns={'data': 'date'})
df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\inflacao_stagging.parquet'").df()
print(json.dumps(list(df.columns)))

["date", "ipca", "igpm"]


In [67]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_custo_producao_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_custo_producao_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df[df["produto"] == "ARROZ                                   "]
df = df.drop(columns=["produto"])
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

df["date"] = pd.to_datetime(
    df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2) + "-01"
).dt.date

df = df.drop(columns=["ano","mes","ano_mes"])

df = df.rename(columns={'vlr_custo_variavel_ha': 'value_variavel',
                        'vlr_custo_variavel_unidade': 'value_variavel_uni',
                        'vlr_custo_fixo_ha': 'value_fixo',
                        'vlr_custo_fixo_unidade': 'value_fixo_uni',
                        'vlr_renda_fator_ha': 'value_renda',
                        'vlr_renda_fator_unidade': 'value_renda_uni'
                        })

df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_custo_producao_stagging.parquet'").df()
## df.head()
print(json.dumps(list(df.columns)))


["empreendimento", "id_produto", "safra", "uf", "municipio", "cod_ibge", "unidade_comercializacao", "value_variavel", "value_variavel_uni", "value_fixo", "value_fixo_uni", "value_renda", "value_renda_uni", "date"]


In [68]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_estimativa_graos_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_estimativa_graos_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df[df["produto"] == "ARROZ                                   "]
df = df.drop(columns=["produto"])
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

ano_inicial = df["ano_agricola"].astype(str).str[:4]
df["date"] = pd.to_datetime(ano_inicial + "-01-01").dt.date
df = df.drop("ano_agricola", axis=1)

df = df.rename(columns={'area_plantada_mil_ha': 'area_plantada',
                        'producao_mil_t': 'producao',
                        'produtividade_mil_ha_mil_t': 'produtividade'
                        })

df.to_parquet(caminho_stagging, index=False)

# duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_estimativa_graos_stagging.parquet'").df()
#df.head()
print(json.dumps(list(df.columns)))


["safra", "uf", "id_produto", "id_levantamento", "dsc_levantamento", "area_plantada", "producao", "produtividade", "date"]


In [69]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_estoque_publico_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_estoque_publico_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df[df["produto"] == "ARROZ                    "]
#Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

df["date"] = pd.to_datetime(
    df["num_ano"].astype(str) + "-" + df["num_mes"].astype(str).str.zfill(2) + "-01"
).dt.date

df = df.drop(columns=["produto","num_ano","num_mes"])

df = df.rename(columns={'qtd_estoque_kg': 'estoque'
                        })

df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_estoque_publico_stagging.parquet'").df()
df.head()


,id_produto,nom_municipio,cod_ibge,uf,conta_operacional,estoque,date
0,4693,RECIFE-PE,2611606,PE,ESTRATÉGIC,"22500,0",2022-01-01
1,4693,BERNARDINO DE CAMPOS-SP,3506300,SP,ESTRATÉGIC,"44100,0",2022-01-01
2,4693,SÃO LUÍS-MA,2111300,MA,ESTRATÉGIC,"37920,0",2022-01-01
3,4693,CRATEÚS-CE,2304103,CE,ESTRATÉGIC,"138960,0",2022-01-01
4,4693,CACHOEIRO DE ITAPEMIRIM-ES,3201209,ES,ESTRATÉGIC,"30900,0",2022-01-01


In [70]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_frete_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_frete_stagging.parquet"
df = pd.read_parquet(caminho_raw)
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

df["date"] = pd.to_datetime(
    df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2) + "-01"
).dt.date

df = df.drop(columns=["ano","mes"])

df = df.rename(columns={'distanicia_km': 'distancia',
                        'valor_frete_tonelada': 'value_weight',
                        'valor_tonelada_km': 'value(weight/distance)'
                        })


df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_frete_stagging.parquet'").df()
# df.head()
print(json.dumps(list(df.columns)))

["dsc_fonte", "municipio_origem", "cod_ibge_origem", "uf_origem", "municipio_destino", "cod_ibge_destino", "uf_destino", "distancia_km", "value_weight", "value(weight/distance)", "date"]


In [71]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_oferta_demanda_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_oferta_demanda_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df[df["produto"] == "ARROZ                                   "]
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

ano_inicial = df["dsc_safra"].astype(str).str[:4]
df["date"] = pd.to_datetime(ano_inicial + "-01-01").dt.date

df = df.drop(columns=["produto","dsc_safra"])

df = df.rename(columns={'estoque_inicial_1000t': 'estoque_inicial',
                        'producao_1000t': 'producao',
                        'importacao_1000t': 'importacao',
                        'consumo_1000t': 'consumo',
                        'exportacao_1000t': 'exportacao',
                        'estoque_final_1000t': 'estoque_final'
                        })

df.to_parquet(caminho_stagging, index=False)

# duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_oferta_demanda_stagging.parquet'").df()
#df.head()
print(json.dumps(list(df.columns)))


["id_produto", "estoque_inicial", "producao", "importacao", "consumo", "exportacao", "estoque_final", "date"]


In [72]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_preco_uf_mensal_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_preco_uf_mensal_stagging.parquet"
df = pd.read_parquet(caminho_raw)
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

df["date"] = pd.to_datetime(
    df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2) + "-01"
).dt.date

df = df.drop(columns=["ano","mes"])

df = df.rename(columns={'valor_produto_kg': 'value'
                        })

df.to_parquet(caminho_stagging, index=False)

##duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_preco_uf_mensal_stagging.parquet'").df()

#df.head()
print(json.dumps(list(df.columns)))


["produto", "classificao_produto", "id_produto", "uf", "regiao", "dsc_nivel_comercializacao", "value", "date"]


In [73]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_preco_municipio_mensal_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_preco_municipio_mensal_stagging.parquet"
df = pd.read_parquet(caminho_raw)
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns


# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

df["date"] = pd.to_datetime(
    df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2) + "-01"
).dt.date

df = df.drop(columns=["ano", "mes"])

df = df.rename(columns={'valor_produto_kg': 'value'
                        })

df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_preco_municipio_mensal_stagging.parquet'").df()

#df.head()
print(json.dumps(list(df.columns)))

["produto", "classificao_produto", "id_produto", "nom_municipio", "cod_ibge", "uf", "regiao", "dsc_nivel_comercializacao", "value", "date"]


In [74]:
caminho_raw = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Raw\conab_serie_historica_graos_raw.parquet"
caminho_stagging = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_serie_historica_graos_stagging.parquet"
df = pd.read_parquet(caminho_raw)
df = df[df["produto"] == "ARROZ                                   "]
# Seleciona as colunas do tipo objeto/string
colunas_texto = df.select_dtypes(include=['object', 'string']).columns

# Aplica .str.strip() em todas as colunas de texto de uma vez
df[colunas_texto] = df[colunas_texto].apply(lambda x: x.str.strip())

ano_inicial = df["ano_agricola"].astype(str).str[:4]
df["date"] = pd.to_datetime(ano_inicial + "-01-01").dt.date

df = df.drop(columns=["produto","ano_agricola"])

df = df.rename(columns={'area_plantada_mil_ha': 'area_plantada',
                        'producao_mil_t': 'producao',
                        'produtividade_mil_ha_mil_t': 'produtividade'
                        })

df.to_parquet(caminho_stagging, index=False)

#duckdb.sql("DESCRIBE SELECT * FROM 'D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging\conab_serie_historica_graos_stagging.parquet'").df()
#df.head()
print(json.dumps(list(df.columns)))


["dsc_safra_previsao", "uf", "id_produto", "area_plantada", "producao", "produtividade", "date"]


## Analytics.db

In [3]:
# 1. Defina o local exato onde o banco de dados .duckdb será salvo
# Exemplo Windows: 'C:/meus_projetos/bancos/meu_banco.duckdb'
# Exemplo Linux/Mac: '/home/usuario/bancos/meu_banco.duckdb'
caminho_banco = "D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb"

# Garante que as pastas do caminho existam antes de criar o arquivo
os.makedirs(os.path.dirname(caminho_banco), exist_ok=True)

# Conecta/Cria o banco no local especificado
con = duckdb.connect(caminho_banco)

# 2. Local onde estão seus arquivos .parquet
diretorio_parquet = "D:\Arquivos Pessoais\Dados\Case Arroz\Dados\Stagging"

# 3. Processa e cria as tabelas
for arquivo in glob.glob(os.path.join(diretorio_parquet, '*.parquet')):
    nome_tabela = os.path.splitext(os.path.basename(arquivo))[0]
    nome_tabela_limpo = nome_tabela.replace('-', '_').replace(' ', '_')
    
    con.execute(f"CREATE OR REPLACE TABLE {nome_tabela_limpo} AS SELECT * FROM '{arquivo}'")
    print(f"Tabela '{nome_tabela_limpo}' criada no banco em: {caminho_banco}")

con.close()

Tabela 'cbot_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'cepea_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_custo_producao_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_estimativa_graos_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_estoque_publico_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_frete_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_oferta_demanda_stagging' criada no banco em: D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb
Tabela 'conab_preco_municipio_mensal_stagging' criada no banco em: D:/Arquivos Pessoa

In [4]:
con = duckdb.connect("D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/analytics.duckdb")

print("Banco DuckDB criado com sucesso!")

con.close()

Banco DuckDB criado com sucesso!


In [5]:
# Conecte ao seu arquivo
con = duckdb.connect("D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/analytics.duckdb", read_only=True)

# 1. Listar todas as tabelas
tabelas = con.execute("SHOW TABLES").fetchall()
print("Tabelas encontradas:", tabelas)

# 2. Caso use schemas personalizados, liste todas as tabelas de todos os schemas
todas_tabelas = con.execute(
    "SELECT table_schema, table_name FROM information_schema.tables"
).df()
print(todas_tabelas)

con.close()

Tabelas encontradas: []
Empty DataFrame
Columns: [table_schema, table_name]
Index: []


In [6]:
import duckdb

con = duckdb.connect(
    "D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb",
    read_only=True
)

tabelas = con.execute("SHOW TABLES").fetchall()

print("Tabelas encontradas:")
for tabela in tabelas:
    print(tabela)

con.close()

Tabelas encontradas:
('cbot_stagging',)
('cepea_stagging',)
('conab_custo_producao_stagging',)
('conab_estimativa_graos_stagging',)
('conab_estoque_publico_stagging',)
('conab_frete_stagging',)
('conab_oferta_demanda_stagging',)
('conab_preco_municipio_mensal_stagging',)
('conab_preco_uf_mensal_stagging',)
('conab_serie_historica_graos_stagging',)
('cooperativa_stagging',)
('diesel_stagging',)
('dolar_stagging',)
('inflacao_stagging',)


In [7]:
import duckdb

caminho_banco = "D:/Arquivos Pessoais/Dados/Case Arroz/Dados/Banco_dados/stg/analytics.duckdb"

con = duckdb.connect(caminho_banco, read_only=True)

# Lista as tabelas
tabelas = con.execute("SHOW TABLES").fetchall()

print("Quantidade de linhas por tabela:\n")

for tabela in tabelas:
    nome_tabela = tabela[0]

    quantidade = con.execute(
        f'SELECT COUNT(*) FROM "{nome_tabela}"'
    ).fetchone()[0]

    print(f"{nome_tabela}: {quantidade:,} linhas")

con.close()

Quantidade de linhas por tabela:

cbot_stagging: 6,771 linhas
cepea_stagging: 5,233 linhas
conab_custo_producao_stagging: 100 linhas
conab_estimativa_graos_stagging: 2,824 linhas
conab_estoque_publico_stagging: 2,263 linhas
conab_frete_stagging: 10,404 linhas
conab_oferta_demanda_stagging: 8 linhas
conab_preco_municipio_mensal_stagging: 27,561 linhas
conab_preco_uf_mensal_stagging: 23,622 linhas
conab_serie_historica_graos_stagging: 1,350 linhas
cooperativa_stagging: 8 linhas
diesel_stagging: 256 linhas
dolar_stagging: 5,453 linhas
inflacao_stagging: 260 linhas
